# EDA: Customer Behavior & Retention Signals

This notebook explores transactional behavior to inform retention modeling decisions.

## Questions answered
1. How has sales evolved over time?
2. Which categories and regions drive the most revenue?
3. What transaction patterns indicate repeat purchase potential?

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

DATA_PATH = Path('ecommerce_sales_data.csv')
df = pd.read_csv(DATA_PATH)
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['quantity'] = pd.to_numeric(df.get('quantity', 0), errors='coerce')
df['sales'] = pd.to_numeric(df.get('sales', 0), errors='coerce')

print('Rows:', len(df))
print('Columns:', list(df.columns))
df.head()

In [ ]:
missing = df.isna().mean().sort_values(ascending=False).rename('missing_rate')
duplicates = df.duplicated().sum()

print('Duplicate rows:', duplicates)
display(missing.to_frame().head(10))

In [ ]:
daily_sales = (
    df.dropna(subset=['order_date'])
      .groupby('order_date', as_index=False)['sales']
      .sum()
      .sort_values('order_date')
)

plt.figure(figsize=(12, 4))
sns.lineplot(data=daily_sales, x='order_date', y='sales')
plt.title('Daily Sales Trend')
plt.xlabel('Order Date')
plt.ylabel('Sales')
plt.tight_layout()
plt.show()

In [ ]:
if 'category' in df.columns:
    cat_sales = (
        df.groupby('category', as_index=False)['sales']
          .sum()
          .sort_values('sales', ascending=False)
          .head(10)
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(data=cat_sales, x='sales', y='category')
    plt.title('Top 10 Categories by Sales')
    plt.tight_layout()
    plt.show()

if 'region' in df.columns:
    region_sales = (
        df.groupby('region', as_index=False)['sales']
          .sum()
          .sort_values('sales', ascending=False)
    )
    display(region_sales)

## EDA-to-Model Decisions

- Use recency/frequency/monetary style features because transaction intensity varies strongly over time.
- Keep temporal splits for validation to avoid leakage from future periods.
- Monitor drift in category/region mix because segment composition can shift model performance.